In [0]:
%sql
-- Spec 4.2 (Nov 1 2026): recency from the latest EVENT_CREATED_AT per invitee - booking
-- recency, not event start. No past-only filter: a future booking's creation still counts.
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_meeting_calendly AS (
WITH coaching_clients AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
calendly_events AS (
    SELECT
        INVITEE_NAME,
        INVITEE_EMAIL,
        EVENT_CREATED_AT
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE (INVITEE_NAME IS NOT NULL OR INVITEE_EMAIL IS NOT NULL)
      AND EVENT_CREATED_AT IS NOT NULL
    -- Spec dedup for duplicate EVENT_URI: INVITEE_UPDATED_AT, then EVENT_CREATED_AT, then insert date
    QUALIFY ROW_NUMBER() OVER (PARTITION BY EVENT_URI ORDER BY INVITEE_UPDATED_AT DESC, EVENT_CREATED_AT DESC, bronze_insert_date DESC) = 1
),
-- Identity via the 2_8 spine (silver.user_identity_map): invitee matches by
-- 32-hex hash (array_contains user_32digits) OR real email (array_contains real_emails);
-- lead attribution merges activity-logged and contact-record-owned leads.
calendly_matched AS (
    SELECT lead_id, MAX(EVENT_CREATED_AT) AS last_created
    FROM (
        SELECT DISTINCT
            explode(sp.lead_ids) AS lead_id,
            ce.EVENT_CREATED_AT
        FROM calendly_events ce
        JOIN crm_ingestion.silver.user_identity_map sp
          ON (ce.INVITEE_NAME IS NOT NULL AND array_contains(sp.user_32digits, LOWER(ce.INVITEE_NAME)))
          OR (ce.INVITEE_EMAIL IS NOT NULL AND array_contains(sp.real_emails, LOWER(ce.INVITEE_EMAIL)))
    )
    GROUP BY lead_id
)
SELECT
    cc.lead_id,
    DATEDIFF(CURRENT_DATE, DATE(cm.last_created)) AS days_since_last_meeting_calendly
FROM coaching_clients cc
JOIN calendly_matched cm ON cc.lead_id = cm.lead_id
)